In [5]:
import pandas as pd
import numpy as np
import os
import pickle
from typing import Dict, Any

# --- 모델 하이퍼파라미터 (규칙) 정의 ---

# 1. 서프라이즈 필터 (ARIMA Z-Score 기준)
Z_SCORE_THRESHOLD = 2.0 

# 2. GICS 필터 (과제 3 최적 Sector)
MOST_SENSITIVE_GICS_SECTORS = [35.0] 

# 3. 모멘텀 필터 (PDF 기반: 20일 기준 8% 이상 상승)
MOMENTUM_THRESHOLD = 0.08 

# 4. 거래량 필터 (PDF 기반: 평소 대비 2배 이상 급증)
VOLUME_RATIO_THRESHOLD = 2.0 

# 모델 메타데이터
MODEL_FILE_NAME = "model_ver4.pkl"
OUT_DIR = "../../output/model" 
os.makedirs(OUT_DIR, exist_ok=True)

In [6]:
def get_investment_decision_ver4(
    surprise_z: float, 
    gics_code: float, 
    momentum_rate: float, 
    volume_ratio: float
) -> Dict[str, Any]:
    """
    최종 통합 하이브리드 모델 (Model Version 4).
    ARIMA Z-Score, GICS, Momentum, Volume의 4가지 조건을 모두 충족할 때만 BUY 결정.
    """
    
    # 1. Decision Logic Check (ALL-IN Strategy)
    
    # R1: Surprise Filter (ARIMA)
    is_surprise_ok = (surprise_z > Z_SCORE_THRESHOLD)
    
    # R2: GICS Filter
    is_gics_ok = (gics_code in MOST_SENSITIVE_GICS_SECTORS)
    
    # R3: Momentum Filter (PDF)
    is_momentum_ok = (momentum_rate >= MOMENTUM_THRESHOLD)
    
    # R4: Volume Filter (PDF)
    is_volume_ok = (volume_ratio >= VOLUME_RATIO_THRESHOLD)
    
    
    # 2. Final Decision (Must satisfy all R1 AND R2 AND R3 AND R4)
    if is_surprise_ok and is_gics_ok and is_momentum_ok and is_volume_ok:
        decision = 'BUY'
        # Long-Only 전략이므로 SELL signal은 발생하지 않음
        reason = "ALL CONDITIONS MET: Optimal GICS, Strong Surprise, High Momentum/Volume."
    else:
        decision = 'HOLD'
        
        # Output Reason for HOLD decision
        missing_filters = []
        if not is_surprise_ok: missing_filters.append("Surprise Z-Score")
        if not is_gics_ok: missing_filters.append("GICS Sector")
        if not is_momentum_ok: missing_filters.append("Momentum Rate")
        if not is_volume_ok: missing_filters.append("Volume Ratio")
        
        reason = f"HOLD. Missing Filters: {', '.join(missing_filters)}"


    # 3. Output Assembly (Expected Return must be looked up separately by the API)
    # NOTE: Expected Return value must be pulled from the pre-calculated GICS 35.0 returns table (Post 20D) by the API.
    
    return {
        'decision': decision,
        'expected_20d_return_note': "API must pull Avg_Return_Post_20D for GICS 35.0.",
        'reason': reason
    }

In [7]:
# --- 모델 규칙 저장 (.pkl) ---
model_rules = {
    'model_name': 'Final_Integrated_Classifier_V4',
    'z_threshold': Z_SCORE_THRESHOLD,
    'gics_sectors': MOST_SENSITIVE_GICS_SECTORS, 
    'momentum_threshold': MOMENTUM_THRESHOLD,
    'volume_threshold': VOLUME_RATIO_THRESHOLD,
    'version': 'v4.0 (4-Factor Hybrid)'
}

model_path = os.path.join(OUT_DIR, MODEL_FILE_NAME)
with open(model_path, 'wb') as f:
    pickle.dump(model_rules, f)
    
print(f"\n[OK] 최종 모델 Version 4 규칙 저장 완료: {model_path}")


# --- 모델 테스트 (API 호출 시뮬레이션) ---
print("\n" + "="*80)
print("FINAL MODEL (VERSION 4) TEST CASES")
print("="*80)

# Case 1: ALL PASS (Ideal scenario for a BUY)
test_case_1 = get_investment_decision_ver4(
    surprise_z=2.5,  # PASS ( > 2.0)
    gics_code=35.0,  # PASS
    momentum_rate=0.15, # PASS ( >= 8%)
    volume_ratio=2.5  # PASS ( >= 2.0)
)
print("Case 1 (ALL PASS - BUY):", test_case_1['decision'], "| Reason:", test_case_1['reason'])

# Case 2: Z-SCORE FAIL (Expected HOLD)
test_case_2 = get_investment_decision_ver4(
    surprise_z=1.5,  # FAIL
    gics_code=35.0,  # PASS
    momentum_rate=0.15, # PASS
    volume_ratio=2.5  # PASS
)
print("Case 2 (Z-SCORE FAIL - HOLD):", test_case_2['decision'], "| Reason:", test_case_2['reason'])

print("="*80)


[OK] 최종 모델 Version 4 규칙 저장 완료: ../../output/model/model_ver4.pkl

FINAL MODEL (VERSION 4) TEST CASES
Case 1 (ALL PASS - BUY): BUY | Reason: ALL CONDITIONS MET: Optimal GICS, Strong Surprise, High Momentum/Volume.
Case 2 (Z-SCORE FAIL - HOLD): HOLD | Reason: HOLD. Missing Filters: Surprise Z-Score


In [4]:
# --- 모델 규칙 저장 (.pkl) ---
model_rules = {
    'model_name': 'Final_Integrated_Classifier_V4',
    'z_threshold': Z_SCORE_THRESHOLD,
    'gics_sectors': MOST_SENSITIVE_GICS_SECTORS, 
    'momentum_threshold': MOMENTUM_THRESHOLD,
    'volume_threshold': VOLUME_RATIO_THRESHOLD,
    'version': 'v4.0 (4-Factor Hybrid)'
}

model_path = os.path.join(OUT_DIR, MODEL_FILE_NAME)
with open(model_path, 'wb') as f:
    pickle.dump(model_rules, f)
    
print(f"\n[OK] 최종 모델 Version 4 규칙 저장 완료: {model_path}")


# --- 모델 테스트 (API 호출 시뮬레이션) ---
print("\n" + "="*80)
print("FINAL MODEL (VERSION 4) TEST CASES")
print("="*80)

# Case 1: ALL PASS (Ideal scenario for a BUY)
test_case_1 = get_investment_decision_ver4(
    surprise_z=2.5,  # PASS ( > 2.0)
    gics_code=35.0,  # PASS
    momentum_rate=0.15, # PASS ( >= 8%)
    volume_ratio=2.5  # PASS ( >= 2.0)
)
print("Case 1 (ALL PASS - BUY):", test_case_1['decision'], "| Reason:", test_case_1['reason'])

# Case 2: Z-SCORE FAIL (Expected HOLD)
test_case_2 = get_investment_decision_ver4(
    surprise_z=1.5,  # FAIL
    gics_code=35.0,  # PASS
    momentum_rate=0.15, # PASS
    volume_ratio=2.5  # PASS
)
print("Case 2 (Z-SCORE FAIL - HOLD):", test_case_2['decision'], "| Reason:", test_case_2['reason'])

print("="*80)


[OK] 최종 모델 Version 4 규칙 저장 완료: ../../output/model/model_ver4.pkl

FINAL MODEL (VERSION 4) TEST CASES
Case 1 (ALL PASS - BUY): BUY | Reason: ALL CONDITIONS MET: Optimal GICS, Strong Surprise, High Momentum/Volume.
Case 2 (Z-SCORE FAIL - HOLD): HOLD | Reason: HOLD. Missing Filters: Surprise Z-Score
